# Exploratory Analysis PSDO

In [ ]:
# Impot libraries
import pandas as pd
from pathlib import Path
import os
import matplotlib.pyplot as plt

In [ ]:
# Import processed data from csv
# Define the path to the processed data based on the current user
current_user = os.getlogin()
if current_user == "bouba.ismalia":
    PROC_DIR = Path(rf"C:\Users\{current_user}\Stichting Hogeschool Utrecht\FCA-DA-P - data\processed")
else:
    PROC_DIR = Path(rf"C:\Users\{current_user}\Stichting Hogeschool Utrecht\FCA-DA-P - Analytics\Student drop-out\data\processed")

# Read the processed data
data = pd.read_csv(PROC_DIR / "v2_combined.csv")

## Analyse SKC relation with drop-out

In [ ]:
# Analyse relation of sdo2_skc_ADVIES_DEF with drop-out, and include missing data
# Include number of students who dropped out per SKC advice category
data['sdo2_skc_ADVIES_DEF'] = data['sdo2_skc_ADVIES_DEF'].fillna('Missing')
data['sdo5_degree_drop_out'] = data['sdo5_degree_drop_out'].fillna('Missing')
sdo_skc_dropout = data.groupby('sdo2_skc_ADVIES_DEF')['sdo5_degree_drop_out'].value_counts(normalize=True).unstack().fillna(0) * 100

# Total number of students per SKC advice category, with percentages
print(data['sdo2_skc_ADVIES_DEF'].value_counts(normalize=True) * 100)

# First look at how many students dropped out per SKC advice category
drop_out_per_SKC_collegeyear = data.groupby('sdo2_skc_ADVIES_DEF')['sdo5_degree_COLLEGEJAAR'].value_counts().unstack().fillna(0)
print(drop_out_per_SKC_collegeyear)

# Visualize as a line graph: each advice type is a line over the college years
drop_out_per_SKC_collegeyear.T.plot(kind='line', marker='o', figsize=(10, 6))
plt.title('Aantal studenten per SKC advies over collegejaren')
plt.ylabel('Aantal studenten')
plt.xlabel('Collegejaar')
plt.legend(title='SKC Advies', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

# Some advice categories have very few students, exclude them from significant analysis
# Exclude studies with less than 100 students in that category
sdo_skc_counts = data['sdo2_skc_ADVIES_DEF'].value_counts()
valid_skc_categories = sdo_skc_counts[sdo_skc_counts >= 200].index
sdo_skc_dropout = sdo_skc_dropout.loc[valid_skc_categories]

# Print the drop-out rates by SKC advice category
print(sdo_skc_dropout)

# Visualize the results
sdo_skc_dropout.plot(kind='bar', stacked=True, figsize=(10, 6), title='Drop-out Rate by SKC Advice Category')
plt.ylabel('Percentage')
plt.xlabel('SKC Advies')
plt.legend(title='Drop-out Status', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

# Are any of the differences significant?
from scipy.stats import chi2_contingency
contingency_table = pd.crosstab(data['sdo2_skc_ADVIES_DEF'], data['sdo5_degree_drop_out'])
chi2, p, dof, expected = chi2_contingency(contingency_table)
print(f"Chi-squared: {chi2}, p-value: {p}")

# Exlude 'Missing' category for significance testing
sdo_skc_dropout = sdo_skc_dropout.drop(index='Missing', errors='ignore')

# Print the drop-out rates by SKC advice category
print(sdo_skc_dropout)

# Visualize the results
sdo_skc_dropout.plot(kind='bar', stacked=True, figsize=(10, 6), title='Drop-out Rate by SKC Advice Category')
plt.ylabel('Percentage')
plt.xlabel('SKC Advies')
plt.legend(title='Drop-out Status', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

# Are any of the differences significant?
#Exclude categories with less than 100 students
data_no_small = data[data['sdo2_skc_ADVIES_DEF'].isin(valid_skc_categories)]

#Exclude 'Missing' category for significance testing
data_no_missing = data_no_small[data_no_small['sdo2_skc_ADVIES_DEF'] != 'Missing']
from scipy.stats import chi2_contingency
contingency_table = pd.crosstab(data_no_missing['sdo2_skc_ADVIES_DEF'], data['sdo5_degree_drop_out'])
chi2, p, dof, expected = chi2_contingency(contingency_table)
print(f"Chi-squared: {chi2}, p-value: {p}")

### Results
SKC Advice type has a strong significant relationship with HU freshman drop-out (p << 0.001). Interestingly, a lot of students (10.7%) that start a degree do not have any SKC advice type (i.e. `Missing` advice type) and this group of students shows the lowest drop-out rates (23.3%). When we exclude this SKC advice type, the strong significant effect between SKC advice type and HU freshman drop-out remains (p << 0.001), indicating that the 'actual' SKC advice types also have a strong relation with drop-out. 

Highest drop-out was observed for students with a `Negatief` advice type (45.5 of students drop-out in their first year), followed by `Positief met aandachtspunten` and `Neutraal` (41.4% and 34.6% respectively). Lowest drop-out was observed in students that received a `Positief` advice type (32.1%).

## Analyse previous education school name with drop-out

In [ ]:
# Analyse relation of sdo1_previous_education with drop-out, and include missing data
# Include number of students who dropped out per previous education category
data['sdo1_previous_Previous_School'] = data['sdo1_previous_Previous_School'].fillna('Missing')
print(data.groupby('sdo1_previous_Previous_School')['sdo5_degree_drop_out'].value_counts(normalize=True).unstack().fillna(0) * 100)

# Find all previous education categories with at least 100 students
sdo_prev_counts = data['sdo1_previous_Previous_School'].value_counts()
valid_prev_categories = sdo_prev_counts[sdo_prev_counts >= 100].index

# Select only rows with previous education categories > 100 students
data_prev_valid = data[data['sdo1_previous_Previous_School'].isin(valid_prev_categories)]
# Exclude 'Missing' category for significance testing
data_prev_valid = data_prev_valid[data_prev_valid['sdo1_previous_Previous_School'] != 'Missing']
# Print the drop-out rates by previous education category
print(data_prev_valid.groupby('sdo1_previous_Previous_School')['sdo5_degree_drop_out'].value_counts())

# Use Fisher's Exact Test to determine if differences are significant, using filtered data
from scipy.stats import fisher_exact
contingency_table_prev = pd.crosstab(data_prev_valid['sdo1_previous_Previous_School'], data_prev_valid['sdo5_degree_drop_out'])
oddsratio, p_value = fisher_exact(contingency_table_prev)
print(f"Odds Ratio: {oddsratio}, p-value: {p_value}")

# Group sdo1_previous_Previous_School into groups based group size
def categorize_school_size(count):
    if count < 10:
        return 1
    elif 10 <= count < 50:
        return 2
    elif 50 <= count < 100:
        return 3
    elif 100 <= count < 500:
        return 4
    else:
        return 5
    
school_size_counts = data['sdo1_previous_Previous_School'].value_counts()
data_regrouped = data.copy()
data_regrouped['Previous_School_Size'] = data['sdo1_previous_Previous_School'].map(school_size_counts).apply(categorize_school_size)
contingency_table_prev = pd.crosstab(data_regrouped['sdo1_previous_Previous_School'], data_regrouped['sdo5_degree_drop_out'])

# Calculate Chi square test
chi2, p, dof, expected = chi2_contingency(contingency_table_prev)
print(f"Chi-squared: {chi2}, p-value: {p}")

contingency_table_size = pd.crosstab(data_regrouped['Previous_School_Size'], data_regrouped['sdo5_degree_drop_out'])

# Test for trend using logistic regression
import statsmodels.api as sm
import numpy as np
X = sm.add_constant(data_regrouped['Previous_School_Size'])
model = sm.Logit(data_regrouped['sdo5_degree_drop_out'], X).fit()
print(model.summary())

# Visualize the results
drop_out_per_prevschool = data_regrouped.groupby('Previous_School_Size')['sdo5_degree_drop_out'].value_counts(normalize=True).unstack().fillna(0) * 100
drop_out_per_prevschool.plot(kind='bar', stacked=True, figsize=(10, 6), title='Drop-out Rate by Previous Education School Size')
plt.ylabel('Percentage')
plt.xlabel('Previous Education School Size')
plt.legend(title='Drop-out Status', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

### Results
There are many different (900) previous education schools (i.e. BRIN codes) in the dataset and a large number of schools only have a small number of students that start a degree at the HU: about 800 schools have seen less than 100 students starting a degree at the HU. We can conclude from this that most students that start a degree at the HU come from a relatively small subset of schools (i.e. 106 instead of 900 in our dataset). 

Since some schools have a small number of students, we've used Fisher's Exact Test for small sample sizes with a cut-off of at least 10 students per school. 

We found that there is a strong association between previous education school and HU freshman drop-out (Odds ratio ~ 1.40 * 10^-182, p = 0.0001), but after correcting for multiple comparisons this not statistically significant (p > 0.05). 

Since using previous education school might lead to profiling, combined with above multiple testing complexity, we grouped schools based on how many students they provided to the HU and tested: one group for less than 10 students, one group for between 10 and 50 students, one group for between 50 and 100 students, one group for betwen 100 and 500 students and one group for more then 500 students. 
We found a very strong signification association between previous education HU student size with respect to HU freshman drop-out (Chi-squared = 1282, p << 0.001). 